## Step 1: Connect to the database

Connect to the local PostgreSQL database running in Docker.

In [5]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection settings
DB_USER = "admin"
DB_PASSWORD = "admin123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"

# Connect to the database
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("✅ Connected to database")

✅ Connected to database


## Step 2: Read All Tables

In [6]:
# List of all tables in the database
table_names = [
    "customers",
    "geolocation",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "category_translation",
]

# Read each table into a dictionary of DataFrames
tables = {}
for name in table_names:
    tables[name] = pd.read_sql_table(name, engine)
    print(f"{name}: {tables[name].shape[0]} rows, {tables[name].shape[1]} columns")

customers: 99441 rows, 5 columns
geolocation: 1000163 rows, 5 columns
orders: 99441 rows, 8 columns
order_items: 112650 rows, 7 columns
order_payments: 103886 rows, 5 columns
order_reviews: 99224 rows, 7 columns
products: 32951 rows, 9 columns
sellers: 3095 rows, 4 columns
category_translation: 71 rows, 2 columns


## Step 3: Check Row Counts and Duplicates
           We check if order_items, order_payments, and order_reviews 
           have more than one row per order — this means we need 
           to aggregate them before joining.

In [7]:
# Preview each table's structure
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.columns.tolist())
    print(df.head(2))


=== customers ===
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  

=== geolocation ===
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  

=

## Step 4: Aggregate order_items
           Since one order can have multiple items, we aggregate 
           to get one row per order (total price, item count).

In [8]:
# Check for duplicate primary keys and unique order_id counts
print("Unique order_id counts per table:\n")
for name, df in tables.items():
    if "order_id" in df.columns:
        total_rows = len(df)
        unique_orders = df["order_id"].nunique()
        print(f"{name}: {total_rows} rows, {unique_orders} unique order_id -> {'⚠️ needs aggregation' if total_rows != unique_orders else '✅ one row per order'}")

Unique order_id counts per table:

orders: 99441 rows, 99441 unique order_id -> ✅ one row per order
order_items: 112650 rows, 98666 unique order_id -> ⚠️ needs aggregation
order_payments: 103886 rows, 99440 unique order_id -> ⚠️ needs aggregation
order_reviews: 99224 rows, 98673 unique order_id -> ⚠️ needs aggregation


## Step 5: Aggregate order_items
An order can contain multiple items, so we aggregate to get one row per order: number of items, total price, and total freight value.

In [9]:
# Aggregate order_items: one row per order
order_items_agg = tables["order_items"].groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

print(f"order_items_agg: {order_items_agg.shape[0]} rows (should match unique order_id count)")
order_items_agg.head()

order_items_agg: 98666 rows (should match unique order_id count)


,order_id,n_items,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


## Step 6: Aggregate order_payments
An order can have multiple payment records (e.g., paid in installments or split across payment methods). We aggregate to get one row per order: total payment value, number of payment installments, and the most common payment type.

In [10]:
# Aggregate order_payments: one row per order
order_payments_agg = tables["order_payments"].groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    n_payments=("payment_sequential", "count"),
    max_installments=("payment_installments", "max")
).reset_index()

print(f"order_payments_agg: {order_payments_agg.shape[0]} rows (should match unique order_id count)")
order_payments_agg.head()

order_payments_agg: 99440 rows (should match unique order_id count)


,order_id,total_payment_value,n_payments,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


## Step 7: Aggregate order_reviews
An order can have more than one review record. We keep the most relevant one per order — typically the latest review score.

In [11]:
# Aggregate order_reviews: one row per order
order_reviews_agg = tables["order_reviews"].groupby("order_id").agg(
    review_score=("review_score", "mean"),
    n_reviews=("review_id", "count")
).reset_index()

print(f"order_reviews_agg: {order_reviews_agg.shape[0]} rows (should match unique order_id count)")
order_reviews_agg.head()

order_reviews_agg: 98673 rows (should match unique order_id count)


,order_id,review_score,n_reviews
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


## Step 8: Build the Final ML Table
Join orders with customers, and the aggregated order_items, order_payments, and order_reviews tables. We use `orders` as the base since it has exactly one row per order. We use LEFT JOIN so we don't lose any orders, even if some have no items, payments, or reviews.

In [12]:
# Build the final ML table: one row per order
ml_table = tables["orders"].merge(
    tables["customers"], on="customer_id", how="left"
).merge(
    order_items_agg, on="order_id", how="left"
).merge(
    order_payments_agg, on="order_id", how="left"
).merge(
    order_reviews_agg, on="order_id", how="left"
)

print(f"ml_table: {ml_table.shape[0]} rows, {ml_table.shape[1]} columns")
print(f"Unique order_id: {ml_table['order_id'].nunique()}")
ml_table.head()

ml_table: 99441 rows, 20 columns
Unique order_id: 99441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_items,total_price,total_freight,total_payment_value,n_payments,max_installments,review_score,n_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,38.71,3.0,1.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,141.46,1.0,1.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,179.12,1.0,3.0,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,72.20,1.0,1.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,28.62,1.0,1.0,5.0,1.0


## Step 9: Save the Artifact
Save the final ML table as a CSV file so the next notebook can load it directly, without re-running the database queries and joins.

In [13]:
import os

# Create an artifacts folder if it doesn't exist
os.makedirs("artifacts", exist_ok=True)

# Save the ML table
ml_table.to_csv("artifacts/ml_table.csv", index=False)

print(f"✅ Saved ml_table.csv with {ml_table.shape[0]} rows and {ml_table.shape[1]} columns")

✅ Saved ml_table.csv with 99441 rows and 20 columns
